In [ ]:
%load_ext autoreload
%autoreload 2

## Initialization

In [ ]:
import os
import sys
import zarr
import glob
import warnings
import numpy as np
import polars as pl
import pandas as pd
import tracksdata as td
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '../src')
# warnings.filterwarnings('ignore')

In [ ]:
from params import *
from data.preparation import load_gt
from utils.plot import *

### Load

In [ ]:
sample_name = "6bba_09961292"

In [ ]:
%%time
zg = zarr.open_group(TRAIN_DIR + f"{sample_name}.zarr")
arr = np.array(zg["0"][0])

In [ ]:
# np.save("arr.npy", arr)
# %%time
# _ = np.load("arr.npy")

In [ ]:
%%time
nodes, edges, estimated_number_of_nodes = load_gt("6bba_09961292")

In [ ]:
nodes.group_by("t").agg(pl.len().alias("n")).sort("t", descending=False).to_pandas().set_index("t").T

### Convert to npy / csv

In [ ]:
save_dir = Path('../input/processed')
save_dir.mkdir(parents=True, exist_ok=True)

os.makedirs(save_dir / "images", exist_ok=True)
os.makedirs(save_dir / "nodes", exist_ok=True)
os.makedirs(save_dir / "edges", exist_ok=True)


In [ ]:
# samples = glob.glob(TRAIN_DIR + "*.zarr")

# metadata = []
# for sample_path in tqdm(samples):
#     sample_name = sample_path.split("/")[-1].split(".")[0]
#     nodes, edges, estimated_number_of_nodes = load_gt(sample_name)

#     zg = zarr.open_group(TRAIN_DIR + f"{sample_name}.zarr")
#     for i in range(zg["0"].shape[0]):
#         arr = np.array(zg["0"][i])
#         np.save(save_dir / "images" / f'{sample_name}_{i}.npy', arr)

#     nodes.to_pandas().to_csv(save_dir / "nodes" / f'{sample_name}.csv', index=False)
#     edges.to_pandas().to_csv(save_dir / "edges" / f'{sample_name}.csv', index=False)
    
#     metadata.append({
#         "sample_name": sample_name,
#         "nodes_path": save_dir / "nodes" / f'{sample_name}.csv',
#         "edges_path": save_dir / "edges" / f'{sample_name}.csv',
#         "shape": zg["0"].shape,
#         "estimated_number_of_nodes": estimated_number_of_nodes,
#     })

# metadata = pd.DataFrame(metadata)
# metadata.to_csv(INPUT_FOLDER + "metadata.csv", index=False)


### Plot

In [ ]:
nodes = pd.read_csv(NODES_FOLDER + f"{sample_name}.csv")
edges = pd.read_csv(EDGES_FOLDER + f"{sample_name}.csv")


In [ ]:
sample_name = "6bba_09961292"

frame = 0

img = np.load(IMG_FOLDER + f"{sample_name}_{frame}.npy")

In [ ]:
from PIL import Image
save = img[0].astype(np.float32)
save = (save - save.min()) / (save.max() - save.min())
save = (save * 255).astype(np.uint8)
Image.fromarray(save).save("img.png")

In [ ]:
plt.imshow(img[0], cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
for t in np.arange(0, 100, 10):
    img = np.load(IMG_FOLDER + f"{sample_name}_{t}.npy")

    plt.figure(figsize=(50, 10))
    for i, z in enumerate(np.arange(10, 60, 10)):
        plt.subplot(1, 10, i + 1)
        plot_frame(img, nodes, t=t, z=z, tol=1)
    plt.show()

    if t == 10:
        break

## Unsupervised seg

In [ ]:
from tracking_cellmot.segmentation import segment_cells_3d

In [ ]:
sys.path.append("../scripts")
from eval_segmentation import score_frame, match_frame_nodes

In [ ]:
def dist(a, b, scale=(1.625, 0.40625, 0.40625)):
    return np.sqrt(scale[0] ** 2 * (a[0] - b[0]) ** 2 + scale[1] ** 2 * (a[1] - b[1]) ** 2 + scale[2] ** 2 * (a[2] - b[2]) ** 2)

### Sample

In [ ]:
t = 0

sample_name = "6bba_09961292"

img = np.load(IMG_FOLDER + f"{sample_name}_{t}.npy")

nodes = pd.read_csv(NODES_FOLDER + f"{sample_name}.csv")

gt = nodes[nodes["t"] == t][["z", "y", "x"]].values

In [ ]:
labels, seeds = segment_cells_3d(
    img,
    voxel_size=SCALE,
    cell_diameter=6.0,
    # background_diameter=7.0,
    min_cell_diameter=3,
    max_cell_diameter=10,
    min_cell_volume=20,
    seed_prominence=0.05,
    # boundary_fraction=0.,
    min_dist_between_cell=3,
)

print(f'Found {len(seeds)} cells')

In [ ]:
pairs = match_frame_nodes(
    seeds,
    gt,
    t=t,
    scale=SCALE,
    max_distance=7,
)

matched = pairs.to_pandas()[["gt_z", "gt_y", "gt_x"]].values.astype(int)
matched = [tuple(node) for node in matched]

matched_gt = []
unmatched_gt = []
for node in gt:
    if tuple(node) in matched:
        matched_gt.append(node)
    else:
        unmatched_gt.append(node)
matched_gt = np.array(matched_gt)
unmatched_gt = np.array(unmatched_gt)

print(f'-> Matched {len(pairs)} out of {len(gt)} GT nodes')

In [ ]:
mask = np.where(labels > 0, labels, np.nan)
tol = 2

zs = np.unique(unmatched_gt[:, 0]) if len(unmatched_gt) > 0 else [16, 18, 20]

for i in zs:
    plt.figure(figsize=(15, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(img[i], cmap="gray")
    plt.axis("off")
    # break

    centers = seeds[np.abs(seeds[:, 0].astype(int) - i) <= tol]
    kept = mask[i][centers[:, 1], centers[:, 2]] > 0
    centers = centers[kept]
    plt.scatter(
        centers[:, 2],
        centers[:, 1],
        s=30,
        c='dodgerblue',
        edgecolors="white",
        linewidths=0.5,
    )

    gt_centers = gt[gt[:, 0]  == i]
    plt.scatter(
        gt_centers[:, 2],
        gt_centers[:, 1],
        s=25,
        c='green',
    )

    plt.subplot(1, 2, 2)

    mask_plot = mask[i].copy()
    # for idx, v in enumerate(np.unique(mask[i])):
    #     mask_plot = np.where(mask_plot == v, np.random.randint(0, 20), mask_plot)

    plt.imshow(mask_plot, cmap="tab20c")

    gt_centers = gt[gt[:, 0]  == i]
    plt.scatter(
        gt_centers[:, 2],
        gt_centers[:, 1],
        s=25,
        c='green',
    )


    plt.axis("off")

    plt.show()

    # break



In [ ]:
mask = np.where(labels > 0, labels, np.nan)
tol = 3

ids = np.unique(unmatched_gt[:, 0]) if len(unmatched_gt) > 0 else [18]

for i in ids:
    plt.figure(figsize=(7, 7))
    plt.title(f"Z = {i}")

    plt.imshow(img[i], cmap="gray")
    plt.axis("off")
    # break

    if len(unmatched_gt) > 0:
        unmatched_gt_centers = unmatched_gt[unmatched_gt[:, 0] == i]
        plt.scatter(
            unmatched_gt_centers[:, 2],
            unmatched_gt_centers[:, 1],
            s=100,
            color=(0, 0, 0, 0),
            linewidths=2,
            edgecolors='orangered',
            label="Unmatched",
        )

    matched_gt_centers = matched_gt[matched_gt[:, 0] == i] if len(matched_gt) > 0 else []
    if len(matched_gt_centers) > 0:
        plt.scatter(
            matched_gt_centers[:, 2],
            matched_gt_centers[:, 1],
            s=100,
            color=(0, 0, 0, 0),  # (1, 1, 1, 0.5),
            linewidths=2,
            edgecolors='limegreen',
            label="Matched",
        )

        centers = seeds[np.abs(seeds[:, 0].astype(int) - i) <= tol]

        dists_to_gt = []
        for gtc in matched_gt_centers:
            dists_to_gt.append([dist(gtc, c) for c in centers])

        dists_to_gt = np.min(dists_to_gt, axis=0)
        matched = np.argwhere(dists_to_gt <= 7).flatten()
        unmatched = np.argwhere(dists_to_gt > 7).flatten()
    else:
        unmatched = np.arange(len(centers))
        matched = []
    
    if len(matched) > 0:
        plt.scatter(
            centers[matched, 2],
            centers[matched, 1],
            s=20,
            c='skyblue',
            linewidths=1,
            edgecolors='dodgerblue',
            label="Predicted Match",
        )

    if len(unmatched) > 0:
        plt.scatter(
            centers[unmatched, 2],
            centers[unmatched, 1],
            s=10,
            c='orangered',
            linewidths=1,
            edgecolors='red',
            label="Predicted Unmatch",
            alpha=0.5,
        )

    plt.legend()

    plt.show()

    # break



In [ ]:
pairs.filter(pl.col('gt_z') == 18)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(pairs['y'], pairs['x'], s=40, c='skyblue', edgecolors="white", linewidths=0.5, zorder=5, label="Pred")
plt.scatter(pairs['gt_y'], pairs['gt_x'], s=20, c='#eb6834', edgecolors="white", linewidths=0.5, zorder=5, label="GT")
plt.legend()
plt.show()

### Loop

In [ ]:
df = pd.read_csv(INPUT_FOLDER + f"metadata.csv")

In [ ]:
sample_name = "6bba_09961292"

nodes = pd.read_csv(NODES_FOLDER + f"{sample_name}.csv")

In [ ]:
PLOT = True

In [ ]:
for t in tqdm(range(100)):
    img = np.load(IMG_FOLDER + f"{sample_name}_{t}.npy")

    gt = nodes[nodes["t"] == t][["z", "y", "x"]].values

    labels, seeds = segment_cells_3d(
        img,
        voxel_size=SCALE,
        cell_diameter=6.0,
        # background_diameter=7.0,
        min_cell_diameter=3,
        max_cell_diameter=10,
        min_cell_volume=20,
        seed_prominence=0.05,
        # boundary_fraction=0.,
        min_dist_between_cell=3,
    )

    pairs = match_frame_nodes(
        seeds,
        gt,
        t=t,
        scale=SCALE,
        max_distance=7,
    )

    print(f'-> Matched {len(pairs)} out of {len(gt)} GT nodes')

    matched = pairs.to_pandas()[["gt_z", "gt_y", "gt_x"]].values.astype(int)
    matched = [tuple(node) for node in matched]

    matched_gt = []
    unmatched_gt = []
    for node in gt:
        if tuple(node) in matched:
            matched_gt.append(node)
        else:
            unmatched_gt.append(node)
    matched_gt = np.array(matched_gt)
    unmatched_gt = np.array(unmatched_gt)

    if PLOT and len(unmatched_gt):
        
        tol = 3

        for i in np.sort(np.unique(unmatched_gt[:, 0])):
            plt.figure(figsize=(7, 7))
            plt.title(f"Z = {i}")

            plt.imshow(img[i], cmap="gray")
            plt.axis("off")
            # break

            if len(unmatched_gt) > 0:
                unmatched_gt_centers = unmatched_gt[unmatched_gt[:, 0] == i]
                plt.scatter(
                    unmatched_gt_centers[:, 2],
                    unmatched_gt_centers[:, 1],
                    s=100,
                    color=(0, 0, 0, 0),
                    linewidths=2,
                    edgecolors='orangered',
                    label="Unmatched",
                )

            matched_gt_centers = matched_gt[matched_gt[:, 0] == i] if len(matched_gt) > 0 else []
            if len(matched_gt_centers) > 0:
                plt.scatter(
                    matched_gt_centers[:, 2],
                    matched_gt_centers[:, 1],
                    s=100,
                    color=(0, 0, 0, 0),  # (1, 1, 1, 0.5),
                    linewidths=2,
                    edgecolors='limegreen',
                    label="Matched",
                )

                centers = seeds[np.abs(seeds[:, 0].astype(int) - i) <= tol]

                dists_to_gt = []
                for gtc in matched_gt_centers:
                    dists_to_gt.append([dist(gtc, c) for c in centers])

                dists_to_gt = np.min(dists_to_gt, axis=0)
                matched = np.argwhere(dists_to_gt <= 7).flatten()
                unmatched = np.argwhere(dists_to_gt > 7).flatten()
            else:
                unmatched = np.arange(len(centers))
                matched = []
            
            if len(matched) > 0:
                plt.scatter(
                    centers[matched, 2],
                    centers[matched, 1],
                    s=20,
                    c='skyblue',
                    linewidths=1,
                    edgecolors='dodgerblue',
                    label="Predicted Match",
                )

            if len(unmatched) > 0:
                plt.scatter(
                    centers[unmatched, 2],
                    centers[unmatched, 1],
                    s=10,
                    c='orangered',
                    linewidths=1,
                    edgecolors='red',
                    label="Predicted Unmatch",
                    alpha=0.5,
                )

            plt.legend()
            plt.show()
    
        # break


### GT -> Mask

In [ ]:
import math

def make_heatmap(
    pooled_shape: tuple[int, int, int],
    centers_zyx: np.ndarray,
    pool_factor: int=1,
    sigma: float=1.0,
) -> np.ndarray:
    heatmap = np.zeros(pooled_shape, dtype=np.float32)
    if centers_zyx.size == 0:
        return heatmap
    radius = max(1, int(math.ceil(3.0 * sigma)))
    sigma2 = float(sigma) ** 2
    z_max, y_max, x_max = pooled_shape
    for z0, y0, x0 in centers_zyx:
        center = np.array([z0, y0 / pool_factor, x0 / pool_factor], dtype=np.float32)
        zc, yc, xc = [float(v) for v in center]
        z_start = max(0, int(math.floor(zc)) - radius)
        z_stop = min(z_max, int(math.floor(zc)) + radius + 2)
        y_start = max(0, int(math.floor(yc)) - radius)
        y_stop = min(y_max, int(math.floor(yc)) + radius + 2)
        x_start = max(0, int(math.floor(xc)) - radius)
        x_stop = min(x_max, int(math.floor(xc)) + radius + 2)
        if z_start >= z_stop or y_start >= y_stop or x_start >= x_stop:
            continue
        zz = np.arange(z_start, z_stop, dtype=np.float32)[:, None, None]
        yy = np.arange(y_start, y_stop, dtype=np.float32)[None, :, None]
        xx = np.arange(x_start, x_stop, dtype=np.float32)[None, None, :]
        d2 = (zz - zc) ** 2 + (yy - yc) ** 2 + (xx - xc) ** 2
        blob = np.exp(-0.5 * d2 / max(sigma2, 1e-6)).astype(np.float32)
        blob = np.where(blob < 0.1, 0, blob)

        view = heatmap[z_start:z_stop, y_start:y_stop, x_start:x_stop]
        np.maximum(view, blob, out=view)
    return heatmap

In [ ]:
t = 10

img = np.array(zg["0"][t])
mask = make_heatmap(
    img.shape, 
    nodes.filter(pl.col("t") == t)[['z', 'y', 'x']].to_numpy(),
    sigma=3,
)

In [ ]:
z = 10
m = np.where(mask[z] == 0, np.nan, mask[z])
plt.imshow(m)

In [ ]:
plt.figure(figsize=(50, 10))
for i, z in enumerate(np.arange(10, 60, 10)):
    plt.subplot(1, 10, i + 1)
    plot_frame(img, nodes, t=t, z=z, mask=mask)
plt.show()

Done !